# NCEAS Reproducibility.md

## Jisha Goyal, Harshil Chidura, Sukhina Alkhalidy, Sidharth Rao

### 24 March 2026

**Installation**

Our full set of instructions are in our INSTALL.md file on our main Github page. This is the link for it: https://github.com/harshil0217/NCEAS_Unsupervised_NLP/blob/main/INSTALL.md

This link contains the complete instructions for repository cloning, environment configuration, and dataset acquisition. The guide includes the necessary steps to recreate the local development environment and reproduce the project's data processing pipeline.

**Data**

Our research pipeline validates unsupervised NLP methods by comparing modern LLM embeddings against high-quality human-labeled benchmarks.

This project utilizes two primary data streams:

- **Benchmark Datasets**: A source of "Ground Truth" metrics. Includes RCV1-v2 (Reuters news hierarchies), arXiv (30k scientific abstracts), and Amazon Reviews (scalability testing). The predictions made by the clustering algorithms are compared to the actual results of the data.

- **Synthetic Data**: Trajectory-based samples generated via Groq to simulate specific fisheries-related discourse.


Generating 1024-D embeddings takes significant time. To reproduce these figures in seconds rather than hours, we use intermediate checkpoints stored in the NCEAS Teams Data folder.

**1.** Initialize the folders with this command:

   mkdir -p src/data/arxiv src/data/amazon src/data/dbpedia src/data/wos src/data/rcv1_v2
   
**2.** Move the following high-value intermediate files into your local project structure:

    - RCV1: Move rcv1_qwen_metadata.csv and rcv1_qwen_embeddings.npy to src/data/rcv1_v2/.

    - arXiv: Move arxiv_30k_clean.csv to src/data/arxiv/.

    - Amazon: Move train_40k.csv.zip to src/data/amazon/.
  
For the RCV1 raw data, you can also fetch it directly via Python:

      from sklearn.datasets import fetch_rcv1
      
      rcv1 = fetch_rcv1() # Automatically downloads the base benchmark

To verify the installation and data placement, run the demo notebook. This generates a sample figure using the full PHATE benchmark pipeline.
    
    jupyter notebook notebooks/demo.ipynb

The notebook should load data, generate embeddings, and display a clustered 2D visualization without errors.

**Visualizations**

This project uses a wide variety of visualizations across each phase. While we have worked with a myriad of visualizations and graphs throughout this project to display and analyze our findings, only a select number of them were to be displayed in our final result, one of them being our Shepard diagrams. 

Our project utilized a comprehensive suite of visualizations to track findings from Phases 1 to 3. A primary focus of our final analysis is the use of Shepard Diagrams to satisfy reviewer requirements for global structural integrity. By comparing PCA, UMAP, PHATE, and PaCMAP, these diagrams allow us to visually verify that the thematic hierarchy of the news corpus remains intact during dimensionality reduction, providing a transparent link between our unsupervised clusters and the Reuters ground truth.

A primary focus of our final analysis is the use of Shepard Diagrams to satisfy reviewer requirements for global structural integrity. By comparing PCA, UMAP, PHATE, and PaCMAP, these diagrams allow us to visually verify that the thematic hierarchy of the news corpus remains intact during dimensionality reduction.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import os

# 1. SETUP: Define the helper function for Shepard Diagrams
def plot_shepard(x_high, x_low, name, sample_size=500):
    """
    Generates a Shepard Diagram to validate global distance preservation.
    x_high: Original 1024-D embeddings
    x_low: Projected 2D coordinates (PCA, UMAP, etc.)
    """
    # Randomly sample to keep computation under 1 minute
    indices = np.random.choice(len(x_high), sample_size, replace=False)
    
    # Calculate and flatten pairwise distances
    d_high = pairwise_distances(x_high[indices]).flatten()
    d_low = pairwise_distances(x_low[indices]).flatten()

    # Normalize distances to [0, 1] for fair comparison across algorithms
    d_high = d_high / np.max(d_high)
    d_low = d_low / np.max(d_low)
    
    plt.figure(figsize=(6, 6))
    plt.scatter(d_high, d_low, alpha=0.1, s=1, color='teal')

    # Red dashed line represents 'Perfect' linear preservation
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', alpha=0.5)
    
    plt.title(f"Shepard Diagram: {name}")
    plt.xlabel("High-Dimensional Distance (Normalized)")
    plt.ylabel("Low-Dimensional Distance (Normalized)")
    plt.tight_layout()
    
    # Save using standard snake_case filenames for the repo
    filename = f"shepard_{name.lower()}.png"
    plt.savefig(filename, dpi=300)
    plt.show() # Show in notebook for instructor review
    return filename

# 2. EXECUTION: Run the loop for all Phase 3.2 models
# Note: Ensure these .npy files were generated in the previous step
reductions = {
    "PCA": np.load('../data/processed/rcv1_pca_2d.npy'),
    "UMAP": np.load('../data/processed/rcv1_umap_2d.npy'),
    "PHATE": np.load('../data/processed/rcv1_phate_2d.npy'),
    "PaCMAP": np.load('../data/processed/rcv1_pacmap_2d.npy')
}

# x_high is your master 1024-D embedding array from Phase 1.2
x_high = np.load('../data/processed/rcv1_qwen_embeddings.npy')

print("Starting Shepard Diagram generation...")
for name, x_low in reductions.items():
    saved_file = plot_shepard(x_high, x_low, name)
    print(f"Successfully generated and saved: {saved_file}")

The code above reproduces the Shepard Diagrams used to validate the structural integrity of our dimensionality reduction models (PCA, UMAP, PHATE, and PaCMAP). By comparing the normalized pairwise distances of the original 1024-D Qwen3 embeddings against their 2D projections, this code provides mathematical proof that the global thematic relationships of the RCV1-v2 dataset were preserved without significant 'tearing' or distortion.